# Linear Regression by Hand

## Why this notebook comes first

In the next notebook you will fit a line with three lines of scikit-learn and get the answer
immediately. That is the right tool — but it hides the thing worth understanding.

Here we do it the slow way: **you** pick the slope, **you** look at how wrong it is, **you**
try again. By the end you will have found a decent line without any model object at all, and
`LinearRegression` will look like automation of something you already did rather than magic.

> **Where this sits.** This notebook is intuition only — no optimisation algorithm.
> Notebook 1 does the same job with scikit-learn. In the **Gradient Descent** lesson you meet
> the algorithm that automates the searching you are about to do by hand.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## The data

The same 32 cars used throughout this lesson. Heavier cars burn more fuel, so `mpg` should
fall as `weight` rises.

In [ ]:
cars = pd.read_csv("data/cars.csv")
cars[["car_model", "weight", "mpg"]].head()

In [ ]:
X = cars["weight"]
y = cars["mpg"]

plt.scatter(X, y)
plt.xlabel("weight")
plt.ylabel("mpg")
plt.title("Fuel economy against weight");

## A line is two numbers

Every straight line through this cloud is described by exactly two numbers:

$$\hat{y} = \text{intercept} + \text{slope} \times x$$

- **slope** — how much `mpg` changes for each extra unit of `weight`. We expect it negative.
- **intercept** — the predicted `mpg` at `weight = 0`. Not physically meaningful here, but it
  is what fixes the line's height.

Write that as a function so we can try values quickly.

In [ ]:
def predict_by_hand(x, slope, intercept):
    """Predict mpg from weight for a given straight line."""
    return intercept + slope * x

## Guess one: a flat line

Start with the laziest possible model — ignore weight entirely and predict the average `mpg`
for every car. Slope zero, intercept equal to the mean.

In [ ]:
slope, intercept = 0.0, y.mean()

plt.scatter(X, y, label="actual")
plt.plot(X, predict_by_hand(X, slope, intercept), color="magenta", linewidth=2, label="guess")
plt.xlabel("weight")
plt.ylabel("mpg")
plt.legend()
plt.title(f"slope={slope}, intercept={intercept:.1f}");

## How wrong is it?

We need one number that says how bad a line is, otherwise "better" is just an opinion.

Take each car's **residual** — the vertical gap between the real `mpg` and the line — square
it so that over- and under-shooting both count as error, and average across all 32 cars. That
is the **mean squared error**.

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}\left(y_i - \hat{y}_i\right)^2$$

In [ ]:
def mse_by_hand(y_true, y_pred):
    """Mean squared error, written out rather than imported."""
    residuals = y_true - y_pred
    return (residuals ** 2).mean()


mse_by_hand(y, predict_by_hand(X, slope, intercept))

## Guess two: tilt it

Now use the information we ignored. Cars here weigh roughly 1.5 to 5.5 (thousand lbs) and
`mpg` runs from about 10 to 34, so a slope somewhere around −5 is a reasonable first stab.

Try a few and watch the error move.

In [ ]:
for slope_guess in [-2, -4, -5, -6, -8]:
    intercept_guess = y.mean() - slope_guess * X.mean()   # keep the line through the centroid
    error = mse_by_hand(y, predict_by_hand(X, slope_guess, intercept_guess))
    print(f"slope {slope_guess:>3}   intercept {intercept_guess:6.2f}   MSE {error:6.2f}")

Each row is a complete model. The MSE column is the only thing telling you which one to
keep — and that is exactly the judgement `LinearRegression` will make for you.

## Searching more carefully

Trying five values by eye is fine for intuition, but we can sweep a whole range and plot the
error as a function of the slope.

In [ ]:
slopes = np.linspace(-10, 0, 201)
errors = [
    mse_by_hand(y, predict_by_hand(X, s, y.mean() - s * X.mean()))
    for s in slopes
]

plt.plot(slopes, errors)
plt.xlabel("slope")
plt.ylabel("MSE")
plt.title("Error as a function of the slope");

That curve is the whole problem in one picture. It has a **single lowest point**, and
finding it is all that "fitting a linear regression" means.

Read the best slope straight off the sweep:

In [ ]:
best_index = int(np.argmin(errors))
best_slope = slopes[best_index]
best_intercept = y.mean() - best_slope * X.mean()

print(f"best slope     : {best_slope:.3f}")
print(f"best intercept : {best_intercept:.3f}")
print(f"lowest MSE     : {errors[best_index]:.3f}")

In [ ]:
plt.scatter(X, y, label="actual")
plt.plot(X, predict_by_hand(X, best_slope, best_intercept),
         color="magenta", linewidth=2, label="best line found by hand")
plt.xlabel("weight")
plt.ylabel("mpg")
plt.legend();

## What you just did

You defined a model (two numbers), defined a measure of wrongness (MSE), and searched for the
numbers that made the wrongness smallest. That is the entire recipe.

Two things to carry forward:

- **Notebook 1** replaces the search with `LinearRegression().fit(X, y)`, which solves for the
  minimum exactly instead of sweeping. Compare its `.coef_` and `.intercept_` against the
  values you just found — they should be very close, and they differ only because our sweep
  was on a grid and constrained the line through the centroid.
- **The Gradient Descent lesson** replaces the sweep with an algorithm that walks downhill on
  that error curve. When you see the curve there, it is this same curve.

The error curve is also why the method is called **ordinary least squares**: we are choosing
the line with the least sum of squared residuals.